# ============================================================
# Kaggle Notebook: Graph-DKT Ablation — undirected
# ============================================================
# RUN IN ORDER. Each section is a separate cell.
# Expected runtime: ~1.5 hours (DKT retrain replaced with restored weights)
# Accelerator recommendation: GPU T4 x1
#
# ⏱️  DKT weights are RESTORED (not retrained).
# Before running: upload weights/dkt_fold{0-4}.pt to Kaggle as a private
# dataset named 'dkt-weights-5fold', then 'Add Data → Your Datasets'.
# ============================================================

### CELL 0 — Install dependencies


In [ ]:
# Install torch-geometric so the Graph-DKT path is activated.
# The dense GCN uses pure PyTorch (einsum) under the hood, so heavy binary dependencies
# (like pyg_lib, torch_scatter, torch_sparse, torch_cluster) are NOT required.
# Kaggle already has a fully-functional PyTorch + CUDA pre-installed.
# Note: You MUST toggle 'Internet on' in the right panel under Settings to download packages!
try:
    import torch_geometric
    print("torch_geometric is already installed!")
except ImportError:
    print("Installing torch-geometric...")
    !pip install torch-geometric -q || echo "WARNING: Could not install torch-geometric. Check if Internet is toggled ON in the right panel under Settings. Falling back to pure PyTorch dense GCN mode."

!pip install scikit-learn pandas tqdm -q

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    try:
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    except AttributeError:
        props = torch.cuda.get_device_properties(0)
        mem = getattr(props, 'total_memory', getattr(props, 'total_mem', 0))
        print(f"VRAM: {mem / 1e9:.1f} GB")

### CELL 1 — Copy training.csv from Kaggle input


In [ ]:
# When you use Kaggle's "Add Data" → "Upload", the file
# lands in /kaggle/input/<dataset-name>/training.csv

import os, shutil

# Find training.csv in Kaggle input directory
input_dir = "/kaggle/input"
training_csv = None
for root, dirs, files in os.walk(input_dir):
    if "training.csv" in files:
        training_csv = os.path.join(root, "training.csv")
        break

if training_csv is None:
    print("ERROR: training.csv not found in Kaggle input!")
    print("Go to: Add data → Upload → select training.csv")
    print("Then re-run this cell.")
    import sys; sys.exit(1)

print(f"Found: {training_csv}")
print(f"Size: {os.path.getsize(training_csv) / 1e6:.1f} MB")

### CELL 2 — Clone repo and copy training data


In [ ]:
!git clone https://github.com/MannPatel1236/CP-Coach.git /kaggle/working/CP-Coach

# Copy training.csv into repo
data_dir = "/kaggle/working/CP-Coach/backend/data"
os.makedirs(data_dir, exist_ok=True)
shutil.copy(training_csv, os.path.join(data_dir, "training.csv"))

print(f"Copied training.csv to {data_dir}")
!wc -l /kaggle/working/CP-Coach/backend/data/training.csv

### CELL 3 — Verify data + quick stats


In [ ]:
%cd /kaggle/working/CP-Coach/backend

import sys
sys.path.insert(0, "/kaggle/working/CP-Coach/backend")

from data.topic_graph import CPTopicGraph
from training.train_dkt import load_csv_sequences

tg = CPTopicGraph()
# Let's load with max_seq_len=2000 matching our training configurations
seqs = load_csv_sequences("data/training.csv", tg, max_seq_len=2000)
print(f"Loaded {len(seqs)} user sequences (max_seq_len=2000)")

import numpy as np
lengths = np.array([len(s) for s in seqs])
print(f"Sequence length stats (after truncation):")
for p in [50, 75, 90, 95, 99, 100]:
    print(f"  p{p}: {np.percentile(lengths, p):.0f}")

### CELL 4 — Smoke test for graph_dkt --adjacency-mode undirected (3 epochs, 1 fold)

In [ ]:
%cd /kaggle/working/CP-Coach/backend
!python3 -m training.train_dkt \
  --data data/training.csv --model graph_dkt \
  --adjacency-mode undirected \
  --epochs 3 --folds 1 --out weights/ablation_smoke_undirected.pt \
  --batch 16 --device cuda --max-seq-len 2000


### CELL 5 — Restore pre-trained DKT weights (no retrain)

**Why:** DKT retraining takes ~1h per notebook. We've already trained 5-fold DKT
weights elsewhere (the canonical baseline at AUC 0.9674). Upload those once as a
private Kaggle dataset (`dkt-weights-5fold`) and this notebook restores them in seconds.

Saves ~1h × 3 ablation notebooks = ~3h of GPU time across the whole ablation study.

In [ ]:
# Restore pre-trained 5-fold DKT weights from the Kaggle input dataset.
# Dataset name convention: 'dkt-weights-5fold' (upload weights/dkt_fold{0-4}.pt there once).
import glob, os, shutil

os.makedirs('weights', exist_ok=True)
found_any = False
for d in glob.glob('/kaggle/input/dkt-weights-5fold*'):
    for f in glob.glob(os.path.join(d, 'dkt_fold*.pt')):
        shutil.copy(f, 'weights/')
        print(f'Restored: {f}')
        found_any = True

if not found_any:
    raise FileNotFoundError(
        "DKT weights dataset not found in /kaggle/input/.\n"
        "Upload weights/dkt_fold{0,1,2,3,4}.pt as a private Kaggle dataset named\n"
        "'dkt-weights-5fold', then 'Add Data → Your Datasets' before running this cell."
    )

restored = sorted(glob.glob('weights/dkt_fold*.pt'))
print(f'\nRestored {len(restored)} DKT fold weights: {[os.path.basename(f) for f in restored]}')


### CELL 6 — Graph-DKT full retrain, adjacency-mode undirected (3 folds × 50 epochs)

In [ ]:
# Run in foreground so that Kaggle's background commit blocks and runs to completion.
# Expected runtime: ~1.5 hours (DKT retrain replaced with restored weights).

!python3 -m training.train_dkt \
  --data data/training.csv --model graph_dkt \
  --adjacency-mode undirected \
  --epochs 50 --folds 5 --out weights/ablation_undirected.pt \
  --batch 16 --device cuda --max-seq-len 2000


### CELL 7 — Compare results


In [ ]:
# Rename ablation_<mode>_foldN.pt → graph_dkt_foldN.pt so eval_folds.py picks them up.
!for i in 0 1 2 3 4; do cp weights/ablation_undirected_fold${i}.pt weights/graph_dkt_fold${i}.pt; done

# Runs eval_folds.py to load fold0-N weights for DKT vs Graph-DKT,
# evaluates them on their respective validation splits, and prints comparison table.
# CPU evaluation is used by default because the models are tiny (~146K parameters).

!python3 -m training.eval_folds --segment-by-length --device cpu --max-seq-len 2000 --folds 5


### CELL 8 — Download results to your machine


In [ ]:
import zipfile, glob

results = []

# Package ablation_<mode>_foldN.pt weights (the new experimental output)
for f in glob.glob("weights/ablation_undirected_fold*.pt"):
    results.append(f)
    print(f"Packed weight: {f}")

# Package pre-existing DKT fold weights (the canonical baseline)
for f in glob.glob("weights/dkt_fold*.pt"):
    results.append(f)
    print(f"Packed weight: {f}")

# Actually write the zip
with zipfile.ZipFile("/kaggle/working/results.zip", "w") as z:
    for f in results:
        z.write(f)

print(f"\nDone! Download /kaggle/working/results.zip from the Output panel of your Kaggle notebook.")
print(f"Total files in zip: {len(results)}")
